# 04 · Baselines con corte temporal

Pronosticamos cada semana las cuatro siguientes con reglas simples. Las últimas
26 semanas del calendario se reservan como período de test; evaluamos solamente
los orígenes con cuatro respuestas conocidas. No entrenamos modelos en este notebook.

Las etiquetas usan exclusivamente ventas anteriores al test. El promedio se
actualiza en cada fecha de corte con lo conocido hasta esa semana.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

import pandas as pd
from IPython.display import display
from stockifai_forecast.datos import leer_movimientos, agregar_semanal, rellenar_ceros
from stockifai_forecast.features import construir_abt
from stockifai_forecast.evaluacion import dividir_por_fecha, etiquetar_con_train, evaluar_baselines

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)

## 1. Del Excel a la ABT

Sumamos egresos por repuesto y semana, completamos ceros y generamos las features
y los cuatro targets. La preparación conserva todas las filas, incluso las que
todavía no tienen respuestas futuras.

In [3]:
mov = leer_movimientos("../data/raw/Import-Movimientos.xlsx")
semanal = agregar_semanal(mov)
serie = rellenar_ceros(semanal)
abt = construir_abt(serie)
display(pd.DataFrame({
    "tabla": ["movimientos", "semanal", "serie con ceros", "ABT"],
    "filas": [len(mov), len(semanal), len(serie), len(abt)],
}))

,tabla,filas
0,movimientos,70193
1,semanal,28116
2,serie con ceros,616866
3,ABT,616866


## 2. Corte y hueco

`inicio_test` se calcula desde la última semana de la ABT original, sin descontar
el final sin respuestas. Train solo admite filas con `semana + 4 semanas < inicio_test`.
El hueco contiene cuatro semanas de orígenes; no se usa para entrenar ni evaluar.
Las semanas finales sin respuestas son otra exclusión, distinta del hueco.

In [4]:
partes = dividir_por_fecha(abt)
train, test = partes["train"], partes["test"]
inicio_test = partes["inicio_test"]
fin_targets = abt["semana"] + pd.Timedelta(weeks=4)
hueco = abt.loc[(abt["semana"] < inicio_test) & (fin_targets >= inicio_test)]
sin_respuesta = abt.loc[(abt["semana"] >= inicio_test) & abt["target_4"].isna()]
print(f"Inicio del test: {inicio_test:%Y-%m-%d}")
conteos = pd.DataFrame([
    {"parte": nombre, "filas": len(tabla), "semanas": tabla["semana"].nunique(),
     "desde": tabla["semana"].min(), "hasta": tabla["semana"].max()}
    for nombre, tabla in [("train", train), ("hueco", hueco),
                          ("test evaluable", test), ("final sin respuesta completa", sin_respuesta)]
])
assert conteos["filas"].sum() == len(abt)
assert (train["semana"] + pd.Timedelta(weeks=4) < inicio_test).all()
assert test["target_4"].notna().all()
display(conteos)

Inicio del test: 2025-01-27


,parte,filas,semanas,desde,hasta
0,train,503231,229,2020-08-10,2024-12-23
1,hueco,14681,4,2024-12-30,2025-01-20
2,test evaluable,83396,22,2025-01-27,2025-06-23
3,final sin respuesta completa,15558,4,2025-06-30,2025-07-21


## 3. Universo y segmentos fijados con el pasado

La ficha de cada SKU se calcula sobre la serie anterior al test, incluidas las
semanas del hueco. `entra_modelo` exige diez semanas con venta. La frecuencia se
calcula con todas sus semanas, incluidos los ceros. Los grupos de historia son
10–25 y 26+ semanas con venta. Un SKU nuevo en test queda sin ficha y fuera del universo.
El cruce siguiente cuenta repuestos, no filas semanales.

In [5]:
etiquetas = etiquetar_con_train(serie, inicio_test)
universo = etiquetas.loc[etiquetas["entra_modelo"]]
display(pd.crosstab(universo["segmento"], universo["grupo_historia"])
        .reindex(columns=["10-25", "26+"], fill_value=0))
print(f"Repuestos con historia previa: {len(etiquetas)}")
print(f"Repuestos del universo: {len(universo)}")

train_etiquetado = train.merge(etiquetas, on="repuesto", how="left", validate="many_to_one")
test_etiquetado = test.merge(etiquetas, on="repuesto", how="left", validate="many_to_one")
es_universo = test_etiquetado["entra_modelo"].eq(True).fillna(False)
print(f"Filas de test del universo: {es_universo.sum()}")
print(f"Filas de test fuera del universo: {(~es_universo).sum()}")
print(f"SKUs de test sin historia previa: {test_etiquetado.loc[test_etiquetado['entra_modelo'].isna(), 'repuesto'].nunique()}")

Repuestos con historia previa: 3691
Repuestos del universo: 456
Filas de test del universo: 10032
Filas de test fuera del universo: 73364
SKUs de test sin historia previa: 183


grupo_historia,10-25,26+
segmento,,
frecuencia_alta,1,93
intermitente,250,112


## 4. MAE por horizonte, segmento e historia

`cero` predice 0; `prom_8` predice el promedio de ocho semanas en los cuatro
horizontes. El MAE es el promedio de errores absolutos en unidades por SKU-semana:
si vendió 4 y predijimos 1, el error de esa fila es 3.

`n_candidatas` cuenta filas del universo en la celda; `n_filas`, las usadas.
`n_excluidas` cuenta promedios ausentes con respuesta conocida. `n_sin_target`
informa respuestas ausentes por separado (debería ser 0 tras nuestro corte).
Una celda sin observaciones tiene MAE NaN, nunca 0. Los totales son sobre filas,
no promedios de los MAE de segmentos. Los desgloses son vistas alternativas:
no se suman segmento, historia y total.

In [6]:
metricas = evaluar_baselines(test_etiquetado)
display(metricas.round({"mae": 4}))
tabla_mae = metricas.pivot(index=["agrupacion", "grupo", "horizonte"],
                          columns="baseline", values="mae")
display(tabla_mae.round(4))

,baseline,horizonte,agrupacion,grupo,mae,n_candidatas,n_filas,n_excluidas,n_sin_target
0,cero,1,total,total,0.7142,10032,10032,0,0
1,prom_8,1,total,total,0.3808,10032,10032,0,0
2,cero,1,segmento,frecuencia_alta,3.1620,2068,2068,0,0
3,prom_8,1,segmento,frecuencia_alta,1.3140,2068,2068,0,0
4,cero,1,segmento,intermitente,0.0786,7964,7964,0,0
5,prom_8,1,segmento,intermitente,0.1385,7964,7964,0,0
6,cero,1,grupo_historia,10-25,0.0545,5522,5522,0,0
7,prom_8,1,grupo_historia,10-25,0.1003,5522,5522,0,0
8,cero,1,grupo_historia,26+,1.5220,4510,4510,0,0
9,prom_8,1,grupo_historia,26+,0.7243,4510,4510,0,0


baseline                                    cero  prom_8
agrupacion     grupo           horizonte                
grupo_historia 10-25           1          0.0545  0.1003
                               2          0.0556  0.1005
                               3          0.0570  0.0998
                               4          0.0578  0.1016
               26+             1          1.5220  0.7243
                               2          1.4998  0.7300
                               3          1.5031  0.7522
                               4          1.4942  0.7701
segmento       frecuencia_alta 1          3.1620  1.3140
                               2          3.1146  1.3256
                               3          3.1228  1.3744
                               4          3.0948  1.4079
               intermitente    1          0.0786  0.1385
                               2          0.0791  0.1389
                               3          0.0799  0.1383
                               4          0.0826  0.1410
total          total           1          0.7142  0.3808
                               2          0.7048  0.3835
                               3          0.7071  0.3931
                               4          0.7035  0.4022

## 5. Comparación compacta

La siguiente tabla promedia el MAE de los cuatro horizontes con el mismo peso.
La comparación detallada está arriba. Menor MAE significa menor error absoluto;
en demanda intermitente la regla cero puede ganar porque muchas semanas no venden.
Eso no demuestra que sea la mejor política de reposición: aquí no medimos faltantes,
sesgo ni costo de quedarse sin stock. Este resultado no elige todavía el corte del universo.

In [7]:
resumen = (metricas.groupby(["agrupacion", "grupo", "baseline"], dropna=False)["mae"]
           .mean().unstack("baseline"))
resumen["menor_mae"] = resumen.apply(
    lambda fila: "empate" if fila["cero"] == fila["prom_8"]
    else ("cero" if fila["cero"] < fila["prom_8"] else "prom_8"), axis=1
)
display(resumen.round(4))
display(metricas.loc[metricas["agrupacion"] == "total",
                     ["baseline", "horizonte", "n_candidatas", "n_filas", "n_excluidas", "n_sin_target"]])

baseline                          cero  prom_8 menor_mae
agrupacion     grupo                                    
grupo_historia 10-25            0.0562  0.1006      cero
               26+              1.5048  0.7442    prom_8
segmento       frecuencia_alta  3.1235  1.3554    prom_8
               intermitente     0.0800  0.1392      cero
total          total            0.7074  0.3899    prom_8

,baseline,horizonte,n_candidatas,n_filas,n_excluidas,n_sin_target
0,cero,1,10032,10032,0,0
1,prom_8,1,10032,10032,0,0
10,cero,2,10032,10032,0,0
11,prom_8,2,10032,10032,0,0
20,cero,3,10032,10032,0,0
21,prom_8,3,10032,10032,0,0
30,cero,4,10032,10032,0,0
31,prom_8,4,10032,10032,0,0


## Métodos nuevos y decisiones

- `.loc[condición]` filtra filas como `WHERE` en SQL; `.copy()` evita modificar la tabla de entrada.
- `Timedelta(weeks=n)` permite calcular el corte por semanas de calendario, no por cantidad de filas.
- `.notna()` / `.isna()` distinguen respuestas y promedios disponibles de los ausentes.
- `.between(10, 25)` incluye ambos extremos; `.ge(26)` significa mayor o igual a 26.
- `.eq(True).fillna(False)` selecciona el universo y trata etiquetas desconocidas como fuera del universo.
- `.merge(..., validate="many_to_one")` es un LEFT JOIN con control: cada SKU debe tener una sola ficha.
- `crosstab` cuenta repuestos en el cruce de dos categorías, como una tabla dinámica de Excel.
- `.abs().mean()` calcula el MAE: valor absoluto de cada diferencia y luego promedio.
- `.groupby(...)` agrupa resultados por categoría; `.pivot(...)` y `.unstack(...)` ponen las reglas en columnas para compararlas.
- `.nunique()` cuenta valores distintos; `.reindex(...)` fija el orden de las columnas del cruce.
- `.round(4)` solo redondea la presentación; la función conserva la precisión de los cálculos.

La función de evaluación informa los NaN de target por separado, aunque el corte
normal ya los evita. Las categorías vacías no generan filas de segmento/historia;
el total siempre existe. Si se configura un mínimo menor a diez, los elegibles
con menos de diez ventas quedan sin grupo de historia: no se los llama «10–25».
No se imputa `prom_8`, no se redondean sus predicciones y no se cambia ningún módulo previo.
La última semana del Excel se conserva tal como la preparan las funciones existentes.


## Lectura de esta ejecución

El test comienza el **27/01/2025**. Hay **456 repuestos** elegibles según el pasado.

- **10-25** (grupo_historia): cero = 0.0562; promedio de ocho = 0.1006; menor MAE: **cero**.
- **26+** (grupo_historia): cero = 1.5048; promedio de ocho = 0.7442; menor MAE: **prom_8**.
- **frecuencia_alta** (segmento): cero = 3.1235; promedio de ocho = 1.3554; menor MAE: **prom_8**.
- **intermitente** (segmento): cero = 0.0800; promedio de ocho = 0.1392; menor MAE: **cero**.
- **total** (total): cero = 0.7074; promedio de ocho = 0.3899; menor MAE: **prom_8**.

Los números son unidades de error absoluto por repuesto-semana, promediadas entre los cuatro horizontes. Las filas y exclusiones se pueden consultar arriba; no representan porcentajes de acierto.